# E09 — Agent Justification After Stronger Model

Reconstruction-v2. Diagnostic-only: no model calls, no agent build. Determines whether GPT-5-mini's residual failures on frozen A2 justify a selective agent (A3).

In [1]:
import json, os
from pathlib import Path
REPO = Path(os.environ.get('NDATRACE_REPO', Path.cwd()))
E09 = REPO / 'experiments/E09_agent_justification'
E08B = REPO / 'experiments/E08B_stronger_model_diagnostic'
def load(p):
    with open(p) as f:
        return json.load(f)
print('repo:', REPO)

repo: /Users/asmitha/Documents/course/PE6201-EMERGING AI TECHNOLOGIES/Project/ndatrace


## 1. Research question

After upgrading Standard RAG to GPT-5 mini, do the remaining failures contain a meaningful, runtime-observable subset that requires case-dependent information acquisition and therefore justifies a selective agent?

## 2. Why E08B changed the question

E08 (Qwen) concluded B (narrow agent potentially justified) based on Qwen's failure buckets. E08B showed GPT-5-mini resolves 30/31 of Qwen's MODEL_REASONING_LIMITED cases and 19/26 of its AGENTICALLY_FIXABLE cases with zero retrieval/agent change -- so E09 must re-derive justification from GPT's OWN residual failures, not Qwen's.

## 3. Historical agent audit (T-series prior art)

Full inventory in `config.yaml`'s `historical_agent_audit` block. Key finding: the T-series agent's dev-sample inclusion rationale (recovery beating regression ~2:1, p=0.51 then 0.058) **reversed** at the full 2,091-case test set (regression 87 > recovery 65, RAG+agent accuracy fell below plain RAG, p=0.088). Read-only tool/loop patterns are reusable as inspiration; the outcome evidence is a cautionary tale, not a template to imitate.

## 4. The 39 GPT joint failures (verified from raw artifacts)

In [2]:
import csv
rows = list(csv.DictReader(open(E08B / 'results/gpt5mini_failure_analysis.csv')))
cls_wrong = [r for r in rows if r['gold_label'] != r['predicted_label']]
joint_fail = [r for r in rows if r['joint_success'] != 'True']
correct_joint_fail = [r for r in joint_fail if r['gold_label']==r['predicted_label']]
contra_fail = [r for r in joint_fail if r['gold_label']=='Contradiction']
contra_cls_wrong = [r for r in cls_wrong if r['gold_label']=='Contradiction']
print('classification-wrong:', len(cls_wrong))
print('joint-fail:', len(joint_fail))
print('correct-label/joint-fail:', len(correct_joint_fail))
print('Contradiction joint-fail:', len(contra_fail))
print('Contradiction classification-wrong:', len(contra_cls_wrong))
assert len(joint_fail) == 39 and len(cls_wrong) == 32 and len(correct_joint_fail) == 7
assert len(contra_fail) == 15 and len(contra_cls_wrong) == 12

classification-wrong: 32
joint-fail: 39
correct-label/joint-fail: 7
Contradiction joint-fail: 15
Contradiction classification-wrong: 12


## 5. Manual-review methodology

All 39 joint failures manually reviewed (no sampling) -- requirement, GPT prediction/evidence, final top-5 retrieved context, BM25 top-20 candidate pool + ranks (local re-query, zero model calls), gold label, gold evidence text (evaluator-side), Qwen/E05 outcomes for context. Bundle: `scripts/build_e09_residual_review_bundle.py` -> `results/residual_review_bundle.json`. Taxonomy assignment: `scripts/analyze_e09_agent_justification.py`'s `MANUAL_TAXONOMY` dict, with a written evidence-based note per case.

## 6. Failure decomposition (of 39 failures, and of all 150 cases)

In [3]:
import pandas as pd
review = pd.read_csv(E09 / 'results/gpt_residual_failure_analysis.csv')
counts = review['primary_bucket'].value_counts()
df = pd.DataFrame({'count_of_39': counts, 'pct_of_39': (counts/39*100).round(1), 'pct_of_150': (counts/150*100).round(1)})
df

,count_of_39,pct_of_39,pct_of_150
primary_bucket,,,
MODEL_REASONING_LIMITED,26,66.7,17.3
RETRIEVAL_FILTERING_LIMITED,6,15.4,4.0
EVIDENCE_SELECTION_LIMITED,6,15.4,4.0
DYNAMIC_INFORMATION_ACQUISITION,1,2.6,0.7


## 7. Correct-label/joint-fail analysis (7 cases)

In [4]:
cjf = review[review['classification_correct'] == True]
print(cjf[['case_id','gold_label','primary_bucket','review_note']].to_string(index=False))

           case_id    gold_label              primary_bucket                                                                                                                                                                                               review_note
train::273::nda-10    Entailment  EVIDENCE_SELECTION_LIMITED                     Correct label; evidence overlaps only part of a multi-span gold answer (below the 0.5 joint threshold) -- a partial-coverage evidence-selection issue, not a missing-information one.
train::317::nda-13    Entailment  EVIDENCE_SELECTION_LIMITED                                     Correct label; GPT's quoted evidence is real and source-valid but does not overlap the specific gold span -- picked a different (also plausible) supporting sentence.
 train::318::nda-2 Contradiction  EVIDENCE_SELECTION_LIMITED                                                                                                                                  Correct label; partia

## 8. Contradiction residuals (15 cases)

In [5]:
contra = review[review['gold_label']=='Contradiction']
print(contra['primary_bucket'].value_counts())
contra_wrong = contra[contra['classification_correct']==False]
print(f'\n{len(contra_wrong)}/12 Contradiction classification-wrong cases had gold evidence already in final top-5:',
      contra_wrong['retrieval_contains_gold_in_final_top5'].sum())

primary_bucket
MODEL_REASONING_LIMITED        12
EVIDENCE_SELECTION_LIMITED      2
RETRIEVAL_FILTERING_LIMITED     1
Name: count, dtype: int64

12/12 Contradiction classification-wrong cases had gold evidence already in final top-5: 12


## 9. Static vs dynamic distinction

Only **1 of 39** failures passes the strict 6-criteria agentic-justification test (`train::273::nda-1` -- an explicit cross-reference to undefined 'paragraphs (a) to (c) of the definition of Confidential Information'). All 6 RETRIEVAL_FILTERING_LIMITED cases are STATIC-fixable (raise final top_k) since gold was in the BM25 top-20 pool every time.

## 10. Oracle actions

In [6]:
print(review['oracle_action_needed'].value_counts())

oracle_action_needed
none                      32
expand_final_k             6
follow_cross_reference     1
Name: count, dtype: int64


## 11. Runtime-observable signals

Candidates: exception/carve-out cue, reranker score margin, cross-reference-to-named-provision cue. Evaluated across all 150 cases (not just failures).

## 12. Signal evaluation over all 150

In [7]:
signals = load(E09 / 'results/signal_prevalence_150case.json')
print(json.dumps(signals['cross_reference_to_named_provision_cue'], indent=2))

{
  "signal": "cross_reference_to_named_provision_cue",
  "tp": 1,
  "fp": 14,
  "fn": 0,
  "tn": 135,
  "precision": 0.06666666666666667,
  "recall": 1.0,
  "escalation_rate": 0.1
}


## 13. Negative trigger findings (preserved, not discarded)

In [8]:
print('Exception cue:', json.dumps(signals['exception_carveout_cue'], indent=2))
print('Reranker score margin:', json.dumps(signals['reranker_score_margin'], indent=2))

Exception cue: {
  "prevalence_in_successes": 0.990990990990991,
  "prevalence_in_failures": 1.0,
  "prevalence_overall": 0.9933333333333333,
  "conclusion": "USELESS AS STANDALONE TRIGGER -- present in nearly all cases regardless of outcome, re-confirming E08's finding on Qwen now independently on GPT."
}
Reranker score margin: {
  "mean_success": 1.5840339060570743,
  "mean_failure": 1.7045329549373724,
  "median_success": 1.0016926229000092,
  "median_failure": 1.2881832122802734,
  "conclusion": "NOT DISCRIMINATING -- success and failure distributions largely overlap."
}


## 14. Whole-population opportunity (% of all 150, not just failures)

In [9]:
pop = {'static_retrieval_fixable': 6, 'dynamic_agent': 1, 'model_reasoning_limited': 26, 'evidence_selection_limited': 6}
for k, v in pop.items():
    print(f'{k}: {v}/150 = {v/150:.1%}')

static_retrieval_fixable: 6/150 = 4.0%
dynamic_agent: 1/150 = 0.7%
model_reasoning_limited: 26/150 = 17.3%
evidence_selection_limited: 6/150 = 4.0%


## 15. Static Pipeline Opportunity Ceiling

In [10]:
ceiling = load(E09 / 'results/oracle_agent_opportunity_ceiling.json')
print(json.dumps(ceiling['STATIC_PIPELINE_OPPORTUNITY_CEILING'], indent=2))

{
  "label": "Assumes all 6 RETRIEVAL_FILTERING_LIMITED cases are fixed by a deterministic top-k increase (expand_final_k) -- NOT an agent action.",
  "joint_success_ceiling": 0.78,
  "joint_success_uplift_pp": 4.0,
  "accuracy_ceiling": 0.82
}


## 16. Oracle Agent Opportunity Ceiling (upper bound, NOT expected performance)

In [11]:
print(json.dumps(ceiling['ORACLE_AGENT_OPPORTUNITY_CEILING'], indent=2))

{
  "label": "Upper bound only, NOT an expected result. Assumes the single genuinely DYNAMIC_INFORMATION_ACQUISITION case is magically resolved by an agent and no currently-correct case regresses.",
  "n_dynamic_cases": 1,
  "joint_success_ceiling": 0.7466666666666667,
  "joint_success_uplift_pp": 0.6666666666666667,
  "accuracy_ceiling": 0.7933333333333333
}


## 17. Historical T-series reversal warning

The prior agent looked directionally encouraging on a 67-case dev sample and a 500-case subsample (recovery beating regression ~2:1, p=0.51 -> 0.058) but **reversed** on the full 2,091-case test set: regressions (87) exceeded recoveries (65), and RAG+agent accuracy (77.7%) fell below plain RAG (78.7%), p=0.088. A favorable E09 diagnostic on 150 TRAIN cases is justification to PROTOTYPE, never evidence that an agent improves the final architecture.

## 18. Cost/complexity estimate (planning only, not measured)

For the single genuinely dynamic case's pattern (cross-reference resolution): 1 additional retrieval/tool call per escalated case, roughly 10% escalation rate if gated on the cross-reference cue (15/150, precision only 6.7%), meaning ~15 cases would incur extra latency/cost for a ceiling gain of +0.67pp joint success. Blended cost estimate using observed GPT cost (~$0.0017/case): an extra classify() call per escalated case adds ~$0.0017 x 15 = ~$0.026 for the full 150-case run -- cheap in isolation, but the ROI (ceiling gain per dollar) is far worse than simply raising final top_k (free, deterministic, +4.0pp ceiling).

## 19. Final E09 decision

In [12]:
decision = load(E09 / 'results/e09_decision.json')
print(json.dumps(decision, indent=2))

{
  "decision": "A",
  "decision_label": "A3 NOT JUSTIFIED",
  "rationale": "Of all 39 GPT joint failures (manually reviewed, no sampling), exactly 1 case (2.6% of failures, 0.67% of all 150 cases) passes the strict 6-criteria agentic-justification test. The dominant residual failure mode (26/39 = 66.7% of failures, 17.3% of all 150 cases) is MODEL_REASONING_LIMITED -- necessary evidence was already present and, in most cases, already correctly located/quoted by GPT, but the model still drew the wrong conclusion. No agent action changes what information the model has in these cases; a second investigation step would only be 'think again,' which is explicitly excluded from counting as agentic. The second-largest bucket (6/39 = 15.4% of failures, 4.0% of all 150) is RETRIEVAL_FILTERING_LIMITED -- gold evidence was in the BM25 top-20 pool every time but cut by the reranker before the final top-5 -- a STATIC, deterministic fix (raise final top_k) with a materially larger ceiling (+4.0pp jo

## 20. Minimum E10 scope

**Not applicable — Decision A (A3 NOT JUSTIFIED).** No minimum agent scope is specified, per the explicit instruction to only define this if the decision is B or C. If a future E-series pass changes this (e.g. after the static top-k fix and a downstream GPT-prompt experiment reduce/reshape the residual and a materially larger dynamic subset emerges), E10's scope should start from the single confirmed pattern here (cross-reference resolution) as its narrowest possible seed, gated on the cross-reference-to-named-provision signal specifically -- not a general-purpose investigative agent.